# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Published: {getattr(metadata, 'datePublished', '')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")
print(f"License: {getattr(metadata, 'license', '')}")
print("Cite as:", getattr(metadata, 'citeAs', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their @ids and their fields

print("Available record sets in the dataset:\n")
recordsets = list(dataset.record_sets())
for rs in recordsets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print(f"  Number of fields: {len(rs.fields)}\n    Field @ids:")
    for field in rs.fields:
        print(f"    - {field.id}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - {col.id}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in dataset.record_sets()]
dataframes = {}
for record_set_id in record_sets:
    print(f'Extracting records from RecordSet {record_set_id}...')
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records.")
    else:
        print(f"No records found for {record_set_id}.")

# Display columns for the first non-empty DataFrame
for rsid in dataframes:
    print(f'Columns in {rsid}:', dataframes[rsid].columns.tolist())
    display(dataframes[rsid].head())
    break  # Only show the first for overview

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose the main record set for analysis (edit as needed)
main_record_set_id = None
for rsid in dataframes:
    main_record_set_id = rsid
    break

if main_record_set_id is not None:
    df = dataframes[main_record_set_id].copy()
    print(f"Selected RecordSet: {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")

    # Try to infer a numeric field for demo; fallback to 'abundance', 'coverage', etc.
    numeric_candidates = [
        c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) or 'abundance' in c or 'coverage' in c or 'mw' in c or 'weight' in c or 'count' in c.lower()
    ]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        numeric_field = df.columns[0]  # fallback

    print(f"Using numeric field: {numeric_field}\n")

    # Show the distribution
    print("Describe numeric field:")
    print(df[numeric_field].describe())

    # Filtering: example threshold at mean
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a likely categorical field
    group_candidates = [c for c in df.columns if 'group' in c.lower() or 'class' in c.lower() or 'type' in c.lower() or 'sample' in c.lower() or 'condition' in c.lower()]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing first 5 groups):")
        display(grouped_df.head())
    else:
        print("No clear group field found for grouping.\n")
else:
    print("No suitable record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If there's a grouping field, make a boxplot
    if group_candidates:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=df[group_candidates[0]], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_candidates[0]}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrated how to load and explore the Mass Spectrometry Analysis of Extracellular Vesicles dataset using the `mlcroissant` library.
* The Croissant schema enabled programmatic access to record sets and fields by `@id`, facilitating EDA and visualization.
* Further domain-specific analyses can be performed after initial data cleaning and summarization steps shown here.